In [ ]:
class Icd9Encoder(BaseEstimator, TransformerMixin):

    def __init__(self, comorbidity_df=None):
        self.comorbidity_df = comorbidity_df  # safe, sklearn cloneable

    def fit(self, X, y):
        df = X.copy().reset_index(drop = True)
        y = y.reset_index(drop = True)
        df['target'] = y.values

        com = self.comorbidity_df   # ACCESS HERE SAFELY

        merged = df[['subject_id','hadm_id','target']].merge(
            com, on=['subject_id','hadm_id'], how='left'
        )

        self.icd_mortality_ = merged.groupby('icd9_code')['target'].mean()
        self.global_mean_ = df['target'].mean()
        return self

    def transform(self, X):
        df = X.copy().reset_index(drop = True)
        com = self.comorbidity_df

        merged = df[['subject_id','hadm_id']].merge(
            com, on=['subject_id','hadm_id'], how='left'
        )

        merged['mortality_proxy'] = merged['icd9_code'].map(self.icd_mortality_)
        merged['mortality_proxy'] = merged['mortality_proxy'].fillna(self.global_mean_)

        agg = merged.groupby(['subject_id','hadm_id']).agg(
            max_mortality=('mortality_proxy','max'),
            mean_mortality=('mortality_proxy','mean'),
            count_comorbidities=('icd9_code','count')
        ).reset_index()

        agg = agg.fillna({
            'max_mortality': self.global_mean_,
            'mean_mortality': self.global_mean_,
            'count_comorbidities': 0
        })

        return df.merge(agg, on=['subject_id','hadm_id'], how='left').reset_index(drop = True)

class IndexResetter(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.reset_index(drop=True)


In [1]:


test_col = "a"
monkeypatch.setattr(fe, "ICD9_DIAGNOSIS", test_col)

result = change_feature_names(df.copy())

assert result.loc[result['id'] == 1, "a"].iloc[0] == "323", "change_features: patient 1 wrong ICD9"
assert result.loc[result['id'] == 2, "a"].iloc[0] == "409", "change_features: patient 2 wrong ICD9"
assert result.loc[result['id'] == 3, "a"].iloc[0] == "TEE", "change_features: patient 3 wrong ICD9"
assert result.loc[result['id'] == 4, "a"].iloc[0].isna() == True, "change_features: patient 4 wrong ICD9"

IndentationError: unexpected indent (2444280031.py, line 8)

In [10]:
df = pd.DataFrame(
        {
            "FEATURE1": [1,2,3,4,5],
            "FEAT_2": [2,3,4,5,6],
            "ICD_FEATURE": ["44323", "54345", "grgrg", "432dw", None]

        }
    )

In [14]:
df.loc[df['FEATURE1'] == 1, 'ICD_FEATURE']

0    44323
Name: ICD_FEATURE, dtype: object

SyntaxError: invalid decimal literal (586542183.py, line 1)

In [18]:
a = "AB"

In [19]:
a.str()

AttributeError: 'str' object has no attribute 'str'